# 01 データ探索 — 取得可能データの確認

yfinance を使って以下のカテゴリのデータが実際に取得できるか確認する。

| カテゴリ | 内容 |
|----------|------|
| ターゲット | 日経225（^N225） |
| マクロ（先物・指数） | 日経225先物、S&P500、NASDAQ、ダウ |
| マクロ（為替） | USD/JPY、EUR/JPY |
| マクロ（コモディティ） | WTI原油、金 |
| マクロ（債券） | 米10年債利回り |
| ミクロ（主要構成銘柄） | ソフトバンクG（9984.T）、ファーストリテイリング（9983.T） |

## 0. 環境セットアップ（Google Colab）

Colab で実行する場合のみ以下のセルを実行する。

In [ ]:
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(get_ipython())

if IN_COLAB:
    # リポジトリ取得
    !git clone https://github.com/Takumi-Itokawa-Finance/Nikkei_Analysis.git
    %cd Nikkei_Analysis

    # Google Drive マウント（data/ の永続化）
    from google.colab import drive
    drive.mount('/content/drive')

    # Drive 側の data ディレクトリをシンボリックリンクで繋ぐ
    DRIVE_DATA = '/content/drive/MyDrive/Nikkei_Analysis/data'
    os.makedirs(DRIVE_DATA, exist_ok=True)
    if not os.path.exists('data'):
        os.symlink(DRIVE_DATA, 'data')

    # 依存ライブラリのインストール
    !pip install -q yfinance pandas-ta

    # src/ をインポートパスに追加
    import sys
    sys.path.insert(0, '/content/Nikkei_Analysis')

    print('Colab セットアップ完了')
else:
    print('ローカル環境で実行中')

In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)

PERIOD = '2y'
INTERVAL = '1d'
DATA_RAW = 'data/raw'

## 1. ターゲット変数：日経225

In [ ]:
nikkei = yf.download('^N225', period=PERIOD, interval=INTERVAL)
print(f'取得行数: {len(nikkei)}')
print(f'期間: {nikkei.index[0].date()} 〜 {nikkei.index[-1].date()}')
display(nikkei.tail())
print('\n欠損値:')
print(nikkei.isnull().sum())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
nikkei['Close'].plot(ax=axes[0], title='Nikkei 225 終値')
nikkei['Volume'].plot(ax=axes[1], title='売買高', color='orange')
plt.tight_layout()
plt.show()

## 2. マクロ指標：先物・海外株指数

In [ ]:
macro_index_tickers = {
    'Nikkei225_Futures': 'NKD=F',
    'SP500':             '^GSPC',
    'NASDAQ':            '^IXIC',
    'DowJones':          '^DJI',
}

results_index = {}
for name, ticker in macro_index_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL)
    results_index[name] = df
    status = '✓' if len(df) > 0 else '✗'
    na_close = df['Close'].isnull().sum() if len(df) > 0 else '-'
    print(f'{status} {name:25s} ticker={ticker:10s} 行数={len(df):4d}  Close欠損={na_close}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for name, df in results_index.items():
    if len(df) > 0:
        normalized = df['Close'] / df['Close'].iloc[0] * 100
        normalized.plot(ax=ax, label=name)
nikkei_norm = nikkei['Close'] / nikkei['Close'].iloc[0] * 100
nikkei_norm.plot(ax=ax, label='Nikkei225', linewidth=2, linestyle='--', color='black')
ax.set_title('各指数の推移（基準=100）')
ax.legend()
plt.tight_layout()
plt.show()

## 3. マクロ指標：為替

In [ ]:
fx_tickers = {
    'USD_JPY':  'JPY=X',
    'EUR_JPY':  'EURJPY=X',
    'GBP_JPY':  'GBPJPY=X',
    'CNY_JPY':  'CNYJPY=X',
}

results_fx = {}
for name, ticker in fx_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL)
    results_fx[name] = df
    status = '✓' if len(df) > 0 else '✗'
    na_close = df['Close'].isnull().sum() if len(df) > 0 else '-'
    print(f'{status} {name:15s} ticker={ticker:12s} 行数={len(df):4d}  Close欠損={na_close}')

In [ ]:
fig, axes = plt.subplots(len(results_fx), 1, figsize=(12, 3 * len(results_fx)), sharex=True)
for ax, (name, df) in zip(axes, results_fx.items()):
    if len(df) > 0:
        df['Close'].plot(ax=ax, title=name)
plt.tight_layout()
plt.show()

## 4. マクロ指標：コモディティ

In [ ]:
commodity_tickers = {
    'WTI_Crude':  'CL=F',
    'Gold':       'GC=F',
    'Silver':     'SI=F',
    'Copper':     'HG=F',
}

results_commodity = {}
for name, ticker in commodity_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL)
    results_commodity[name] = df
    status = '✓' if len(df) > 0 else '✗'
    na_close = df['Close'].isnull().sum() if len(df) > 0 else '-'
    print(f'{status} {name:15s} ticker={ticker:8s} 行数={len(df):4d}  Close欠損={na_close}')

In [ ]:
fig, axes = plt.subplots(len(results_commodity), 1, figsize=(12, 3 * len(results_commodity)), sharex=True)
for ax, (name, df) in zip(axes, results_commodity.items()):
    if len(df) > 0:
        df['Close'].plot(ax=ax, title=name)
plt.tight_layout()
plt.show()

## 5. マクロ指標：債券利回り

In [ ]:
bond_tickers = {
    'US_10Y':  '^TNX',
    'US_2Y':   '^IRX',
    'US_30Y':  '^TYX',
}

results_bond = {}
for name, ticker in bond_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL)
    results_bond[name] = df
    status = '✓' if len(df) > 0 else '✗'
    na_close = df['Close'].isnull().sum() if len(df) > 0 else '-'
    print(f'{status} {name:10s} ticker={ticker:8s} 行数={len(df):4d}  Close欠損={na_close}')

# 注: 日本国債利回りはyfinanceでの取得が不安定なため要別途確認

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for name, df in results_bond.items():
    if len(df) > 0:
        df['Close'].plot(ax=ax, label=name)
ax.set_title('米国債利回り推移')
ax.legend()
plt.tight_layout()
plt.show()

## 6. ミクロ指標：Nikkei 225 主要構成銘柄

In [ ]:
micro_tickers = {
    'FastRetailing':   '9983.T',
    'SoftBank':        '9984.T',
    'Tokyo_Electron':  '8035.T',
    'KDDI':            '9433.T',
    'Shin_Etsu_Chem':  '4063.T',
}

results_micro = {}
for name, ticker in micro_tickers.items():
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL)
    results_micro[name] = df
    status = '✓' if len(df) > 0 else '✗'
    na_close = df['Close'].isnull().sum() if len(df) > 0 else '-'
    print(f'{status} {name:20s} ticker={ticker:8s} 行数={len(df):4d}  Close欠損={na_close}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
for name, df in results_micro.items():
    if len(df) > 0:
        normalized = df['Close'] / df['Close'].iloc[0] * 100
        normalized.plot(ax=ax, label=name)
ax.set_title('主要構成銘柄の推移（基準=100）')
ax.legend()
plt.tight_layout()
plt.show()

## 7. 取得結果サマリー

In [ ]:
all_results = {
    **{'Nikkei225': nikkei},
    **{f'[Index] {k}': v for k, v in results_index.items()},
    **{f'[FX] {k}': v for k, v in results_fx.items()},
    **{f'[Commodity] {k}': v for k, v in results_commodity.items()},
    **{f'[Bond] {k}': v for k, v in results_bond.items()},
    **{f'[Micro] {k}': v for k, v in results_micro.items()},
}

summary = []
for name, df in all_results.items():
    if len(df) > 0:
        summary.append({
            'データ': name,
            '取得可否': '✓',
            '行数': len(df),
            '開始日': df.index[0].date(),
            '終了日': df.index[-1].date(),
            'Close欠損数': df['Close'].isnull().sum(),
        })
    else:
        summary.append({'データ': name, '取得可否': '✗', '行数': 0,
                        '開始日': '-', '終了日': '-', 'Close欠損数': '-'})

summary_df = pd.DataFrame(summary)
display(summary_df)

In [ ]:
close_df = pd.DataFrame({'Nikkei225': nikkei['Close'].squeeze()})

for name, df in {**results_index, **results_fx, **results_commodity, **results_bond, **results_micro}.items():
    if len(df) > 0:
        close_df[name] = df['Close'].squeeze()

print(f'結合後の形状: {close_df.shape}')
print(f'\n欠損率(%):')
print((close_df.isnull().sum() / len(close_df) * 100).round(1))
display(close_df.tail())

## 8. raw データの保存

In [ ]:
os.makedirs(DATA_RAW, exist_ok=True)

# 結合済み Close を保存（後工程で使い回す）
save_path = f'{DATA_RAW}/close_all.csv'
close_df.to_csv(save_path)
print(f'保存完了: {save_path}')

## 9. 次のステップ

上記サマリーを踏まえて以下を判断する。

- 取得できなかった指標 → 代替 ticker の検討、または除外
- 欠損が多い指標 → 補間方針の検討（ffill / 除外）
- 追加すべき指標 → VIX（^VIX）、騰落レシオ など

確認後、`src/data/fetcher.py` としてデータ取得ロジックを関数化する。